# Прогноз выступлений на Q2–Q4 2026

**Метрики:** Выступления, Платные выступления, Оплаты  
**Исторические данные:** Q1 2024 – Q1 2026  
**Прогноз:** Q2, Q3, Q4 2026

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 130

In [ ]:
quarters = [
    'Q1 2024', 'Q2 2024', 'Q3 2024', 'Q4 2024',
    'Q1 2025', 'Q2 2025', 'Q3 2025', 'Q4 2025',
    'Q1 2026',
]

data = {
    'Выступления':         [1595, 8913, 16406, 27472, 9780, 18358, 10499, 28077, 12350],
    'Платные выступления': [1005, 5615, 10336, 17307, 5281,  9913,  5669, 15162,  5070],
    'Оплаты':              [ 365, 2384,  2970,  9668, 1077,  5548,  5212, 10210,  3761],
}

n_hist = len(quarters)
t_hist = np.arange(1, n_hist + 1)

forecast_quarters = ['Q2 2026', 'Q3 2026', 'Q4 2026']
t_fc = np.array([10, 11, 12])
all_quarters = quarters + forecast_quarters

df_hist = pd.DataFrame(data, index=quarters)
df_hist.index.name = 'Квартал'
print('Исторические данные:')
print(df_hist.to_string())

In [ ]:
from scipy.optimize import curve_fit

def quarter_of(t):
    return ((t - 1) % 4) + 1

def trend_season(t, a, b, s2, s3, s4):
    season = np.where(quarter_of(t) == 2, s2,
             np.where(quarter_of(t) == 3, s3,
             np.where(quarter_of(t) == 4, s4, 0.0)))
    return a + b * t + season

results = {}

for metric, values in data.items():
    y = np.array(values, dtype=float)
    p0 = [y.mean(), 100.0, 2000.0, 0.0, 5000.0]
    popt, pcov = curve_fit(trend_season, t_hist, y, p0=p0, maxfev=20000)
    y_hat_hist = trend_season(t_hist, *popt)
    residuals  = y - y_hat_hist
    rmse = np.sqrt(np.mean(residuals**2))
    mape = np.mean(np.abs(residuals / y)) * 100

    y_fc = trend_season(t_fc, *popt)

    J = np.array([
        [1, t, float(quarter_of(t)==2), float(quarter_of(t)==3), float(quarter_of(t)==4)]
        for t in t_fc
    ])
    var_fc = np.array([J[i] @ pcov @ J[i] + rmse**2 for i in range(len(t_fc))])
    ci95   = 1.96 * np.sqrt(var_fc)

    results[metric] = {
        'popt': popt, 'y_hat_hist': y_hat_hist,
        'y_fc': y_fc, 'ci95': ci95,
        'rmse': rmse, 'mape': mape,
    }

print('Модель обучена.')
print()
for metric, res in results.items():
    print(f'  {metric}: RMSE={res["rmse"]:.0f}, MAPE={res["mape"]:.1f}%')

In [ ]:
print('=== Прогноз Q2–Q4 2026 ===')
print(f'{"Метрика":<25} {"Квартал":<10} {"Прогноз":>10} {"От -95%":>10} {"До +95%":>10}')
print('-' * 70)
for metric, res in results.items():
    for i, q in enumerate(forecast_quarters):
        v  = res['y_fc'][i]
        ci = res['ci95'][i]
        lo = max(0, int(round(v - ci)))
        hi = int(round(v + ci))
        print(f'{metric:<25} {q:<10} {int(round(v)):>10,} {lo:>10,} {hi:>10,}'.replace(',', ' '))

In [ ]:
colors = {
    'Выступления':         '#2563EB',
    'Платные выступления': '#16A34A',
    'Оплаты':              '#DC2626',
}

fig, axes = plt.subplots(3, 1, figsize=(13, 14), sharex=True)
fig.suptitle('Прогноз выступлений Q2–Q4 2026', fontsize=16, fontweight='bold', y=0.99)

x_all  = np.arange(len(all_quarters))
x_hist = x_all[:n_hist]
x_fc   = x_all[n_hist:]

for ax, (metric, res) in zip(axes, results.items()):
    color = colors[metric]
    y_obs = np.array(data[metric], dtype=float)

    ax.plot(x_hist, res['y_hat_hist'], color=color, lw=2,
            ls='--', alpha=0.7, label='Модель (история)')
    ax.plot(x_fc, res['y_fc'], 'o-', color=color, lw=2.5, ms=8, label='Прогноз')
    ax.fill_between(
        x_fc,
        np.maximum(0, res['y_fc'] - res['ci95']),
        res['y_fc'] + res['ci95'],
        color=color, alpha=0.15, label='95% ДИ'
    )
    ax.scatter(x_hist, y_obs, color=color, s=60, zorder=5, label='Факт')

    for xi, yi in zip(x_fc, res['y_fc']):
        ax.annotate(f'{int(round(yi)):,}'.replace(',', '\u202f'),
                    xy=(xi, yi), xytext=(0, 10), textcoords='offset points',
                    ha='center', fontsize=10, fontweight='bold', color=color)

    ymin, ymax = ax.get_ylim()
    ax.axvline(x=x_hist[-1] + 0.5, color='gray', ls=':', lw=1.5)
    ax.text(x_hist[-1] + 0.6, ymax * 0.95, 'прогноз →', fontsize=8, color='gray', va='top')

    ax.set_title(metric, fontsize=13, fontweight='bold', pad=6)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'.replace(',', '\u202f')))
    ax.grid(axis='y', alpha=0.3)
    ax.legend(loc='upper left', fontsize=9, framealpha=0.85)

axes[-1].set_xticks(x_all)
axes[-1].set_xticklabels(all_quarters, rotation=35, ha='right', fontsize=10)

plt.tight_layout()
plt.savefig('forecast_2026.png', bbox_inches='tight')
plt.show()
print('График сохранён: forecast_2026.png')

In [ ]:
summary = {}
for metric in data:
    summary[metric] = list(data[metric]) + [int(round(v)) for v in results[metric]['y_fc']]

df_summary = pd.DataFrame(summary, index=all_quarters)
df_summary.index.name = 'Квартал'

print('\nИстория + прогноз (* — прогнозные кварталы):')
header = f"{'':12}  {'Выступления':>16}  {'Платные выступления':>20}  {'Оплаты':>10}"
print(header)
print('-' * len(header))
for i, (idx, row) in enumerate(df_summary.iterrows()):
    tag = '* ' if i >= n_hist else '  '
    print(f"{tag}{idx:<12}  {row['Выступления']:>16,}  {row['Платные выступления']:>20,}  {row['Оплаты']:>10,}".replace(',', ' '))